# Rhea FinGraph — Temporal GNN Training on Kaggle T4 (Step 4)

Trains the **TeMP-TraG-style temporal heterogeneous GNN** (PyTorch Geometric) on the free Tesla T4 GPU (zero MacBook heat):

1. `graph_snapshots` — builds leakage-safe temporal snapshots (yearly buckets) from the parquet splits; node features (8-dim: count/amount/partners/fraud-rate/fraud-volume/avg-amount/partner-density/velocity) come from **strictly-past** history only
2. `train_gnn` — trains the TemporalHeteroGNN (heterogeneous message passing + causal temporal transformer + edge MLP) and a HomogeneousGraphSAGE baseline.

**This run uses `--event-cutoffs 534 568`** — the same calendar month-idx boundaries as the XGBoost baseline (train<2014-07, val 2014-07..2017-05, test>=2017-05), so the GNN scores are a **fair** comparison and the `gnn_scores.parquet` stream is row-count-aligned for honest `--gnn-score-file` fusion with the ensemble.

**Before running:** Session options → Accelerator → **GPU T4 x2** (falls back to CPU automatically if left off), and Input must include your `rhea-fingraph-ibm-splits` dataset (train / validation / test parquets).
**After running:** File → Save Version → **Save & Run All (Commit)**, wait for *Save complete*, then download `rhea_gnn_artifacts.zip` from the Output page.

In [ ]:
%pip install -q -U polars
%pip install -q torch-geometric
%pip install -q --force-reinstall --no-deps git+https://github.com/aditisahu1234/Rhea-FinGraph.git

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

# locate the dataset parquets
by_name = {p.split("/")[-1]: p for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True)}
print("Found:", sorted(by_name))
assert "train.parquet" in by_name and "validation.parquet" in by_name
assert "test.parquet" in by_name

# copy to a predictable layout (the package expects ./data/processed/ibm_full/)
target = Path("data/processed/ibm_full")
target.mkdir(parents=True, exist_ok=True)
for name in ("train.parquet", "validation.parquet", "test.parquet"):
    dest = target / name
    if not dest.exists():
        print(f"copying {by_name[name]} -> {dest} ...", flush=True)
        os.symlink(by_name[name], dest)

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Device: {DEVICE}", flush=True)
if DEVICE == "cpu":
    print(
        "WARNING: no GPU detected -- Session options > Accelerator > GPU T4 x2",
        "and re-run.",
        flush=True,
    )


## 1) Build temporal snapshots (yearly buckets)

~29 snapshots from 1991→2020. Node features are history-only (no leakage).

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.graph_snapshots",
        "--splits", "train", "val", "test",
        "--bucket-months", "12",
        "--out", "/kaggle/working/graph",
    ],
    check=True,
)

## 2) Train the temporal heterogeneous GNN (+ GraphSAGE baseline)

Chronological split: earliest 60% of snapshots = train, next 20% = validation, newest 20% = test (locked).

## 1b) Self-supervised pre-training (label-free)

Masked node-feature reconstruction on the temporal graph (~5 epochs): the model
learns to infer a node's hidden history features from its neighborhood and its
own past periods, **without any fraud labels**. The checkpoint saves only the
embedding side; fine-tuning then starts the edge scorer from scratch on top of
these pre-trained representations.


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.pretrain_gnn",
        "--data-dir", "/kaggle/working/graph",
        "--out", "/kaggle/working/gnn-pretrain",
        "--device", DEVICE,
        "--epochs", "5",
        "--hidden", "64",
        "--layers", "2",
        "--heads", "4",
        "--mask-ratio", "0.3",
    ],
    check=True,
)


In [ ]:
import os

# Strong, event-aligned retrain (honest compare + fusible with XGBoost baseline).
# split_mode=event: partitions snapshots by the SAME calendar month-idx cutoffs
# as the baseline (train<2014-07=534, val 534..568=2017-05, test>=568), so the
# GNN score stream row-counts match the fusion matrix and --gnn-score-file works.
# Richer node features (8-dim, incl. velocity/avg-amount) + real architecture.
train_cmd = [
    sys.executable, "-m", "fingraph_sentinel.train_gnn",
    "--data-dir", "/kaggle/working/graph",
    "--out", "/kaggle/working/gnn",
    "--device", DEVICE,
    "--epochs", "40",
    "--hidden", "192",
    "--layers", "3",
    "--heads", "8",
    "--dropout", "0.2",
    "--lr", "1e-3",
    "--patience", "8",
    "--event-cutoffs", "534", "568",
    "--with-sage",
]
pretrained = os.path.join("/kaggle/working/gnn-pretrain", "gnn_pretrained.pt")
if os.path.exists(pretrained):
    train_cmd += ["--init-from", pretrained]
    print("Fine-tuning from self-supervised pre-trained embeddings.")
subprocess.run(train_cmd, check=True)

## 3) Package artifacts

Download `rhea_gnn_artifacts.zip` from the Output page after the Commit finishes.

In [ ]:
import json
import os
from pathlib import Path

gnn_dir = Path("/kaggle/working/gnn")
config_path = gnn_dir / "gnn_config.json"
if config_path.exists():
    cfg = json.loads(config_path.read_text())
    print("== GNN RESULTS ==")
    print(f"  validation:  {cfg.get('metrics_validation')}")
    print(f"  test locked: {cfg.get('metrics_test_locked')}")
    print(f"  fit_seconds: {cfg.get('fit_seconds')}")
else:
    print("\n!!! NO gnn_config.json in /kaggle/working/gnn — the model was NOT saved.")
    print("    Training likely timed out / OOM'd before finishing.")
    print("    The archive below will NOT contain the trained model, only graph snapshots.")
    print("    Re-run with fewer epochs or WITHOUT --with-sage to fit within the quota,")
    print("    OR split into two notebook runs.")

score_path = gnn_dir / "gnn_scores.parquet"
if gnn_dir.exists() and not (gnn_dir / "gnn_config.json").exists():
    # do not zip a model-less gnn dir: it would silently produce a broken archive
    import shutil
    shutil.rmtree(str(gnn_dir))

zip_targets = "gnn graph" if (gnn_dir / "gnn_config.json").exists() else "graph"
!cd /kaggle/working && zip -qr rhea_gnn_artifacts.zip $zip_targets
!ls -lh /kaggle/working/rhea_gnn_artifacts.zip
print("\nDone. Download rhea_gnn_artifacts.zip from the Output page.")